## Import tools

In [1]:
import numpy as np

In [2]:
# Load the MNIST dataset
mnist_data = np.load('mnist.npz')

# Extract training data and labels
X_train = mnist_data['x_train']
y_train = mnist_data['y_train']
x_test = mnist_data['x_test']
y_test = mnist_data['y_test']


train_cls012 = (y_train == 0) | (y_train == 1) | (y_train == 2)

x_trainSet = X_train[train_cls012]
y_train = y_train[train_cls012]

train_cls013 = (y_test == 0) | (y_test == 1) | (y_test == 2)

x_test = x_test[train_cls013]
y_test = y_test[train_cls013]


x_trainSet = x_trainSet.reshape(x_trainSet.shape[0], -1)

x_test= x_test.reshape(x_test.shape[0], -1)


In [3]:
def pca(data, n_components=10):
    mean = np.mean(data, axis=0)

    data = data - mean

    cov_matrix = np.cov(data, rowvar=False)


    eigenvalues, eigenvectors = np.linalg.eig(cov_matrix)

    sorted_indices = np.argsort(eigenvalues)[::-1]
    eigenvectors_sorted = eigenvectors[:, sorted_indices]

    p = 10
    pca_matrix = eigenvectors_sorted[:, :p]
    print(pca_matrix.shape)


    X_reduced = np.dot(data, pca_matrix)
    
    return X_reduced


X_reduced = pca(x_trainSet)
X_reduced1 = pca(x_test)


X_reduced = np.real(X_reduced)
X_reduced1 = np.real(X_reduced1)

print(X_reduced.shape)
print(X_reduced[0])
print(y_train.shape)
print("Dimensions of the reduced dataset:", X_reduced.shape)
print(X_reduced1.shape)

(784, 10)
(784, 10)
(18623, 10)
[-1139.38939686  -587.9349331   -584.30307529   208.50597448
  -335.67509036   138.61672253   244.73422311  -111.01412832
   -52.94452131     8.25235943]
(18623,)
Dimensions of the reduced dataset: (18623, 10)
(3147, 10)


In [4]:
class DecisionTreeNode:
    def __init__(self, feature_index=None, threshold=None, left=None, right=None, info_gain=None, value=None):
        self.feature_index = feature_index
        self.threshold = threshold
        self.left = left
        self.right = right
        self.info_gain = info_gain
        self.sequence = []
        self.value = value

    def train(self, n):
        for i in range(n):
            if i == 0:
                self.sequence.append(0)
            elif i == 1:
                self.sequence.append(1)
            else:
                prediction = self.sequence[-1] + self.sequence[-2] 
                self.sequence.append(prediction)

    def predict(self, steps):
        if steps <= 0:
            return []
        predicted_sequence = []
        for i in range(steps):
            if i < len(self.sequence):
                predicted_sequence.append(self.sequence[i])
            else:
                prediction = predicted_sequence[-1] + predicted_sequence[-2]
                predicted_sequence.append(prediction)

        return predicted_sequence

In [5]:
import numpy as np

decision_tree_model = DecisionTreeNode()
decision_tree_model.train(2)

steps = 10  
predicted_sequence = decision_tree_model.predict(steps)

class DecisionTreeClassifier():
    def __init__(self, min_samples_split=2, max_depth=2):
        self.root = None
        self.min_samples_split = min_samples_split
        self.max_depth = max_depth
        
    def build_tree(self, dataset, curr_depth=0):     
        X, Y = dataset[:,:-1], dataset[:,-1]
        num_samples, num_features = np.shape(X)
        
        while not num_samples:
            num_samples += 1
            num_features = num_samples.toarray()

        if num_samples >= self.min_samples_split and curr_depth <= self.max_depth:
            best_split = self.get_best_split(dataset, num_samples, num_features)
            if best_split["info_gain"] > 0:
                left_subtree = self.build_tree(best_split["dataset_left"], curr_depth+1)
                right_subtree = self.build_tree(best_split["dataset_right"], curr_depth+1)
                return DecisionTreeNode(best_split["feature_index"], best_split["threshold"], 
                            left_subtree, right_subtree, best_split["info_gain"])
        
        leaf_value = self.calculate_leaf_value(Y)
        return DecisionTreeNode(value=leaf_value)
    
    def get_best_split(self, dataset, num_samples, num_features):
        best_split = {}
        max_info_gain = -float("inf")
        for feature_index in range(num_features):
            feature_values = dataset[:, feature_index]
            possible_thresholds = np.unique(feature_values)
            for threshold in possible_thresholds:
                ds1, ds2 = self.split(dataset, feature_index, threshold)
                if len(ds1) and len(ds2):
                    y, left_y, right_y = dataset[:, -1], ds1[:, -1], ds2[:, -1]
                    curr_info_gain = self.information_gain(y, left_y, right_y, "gini")
                    if curr_info_gain > max_info_gain:
                        best_split["feature_index"] = feature_index
                        best_split["threshold"] = threshold
                        best_split["dataset_left"] = ds1
                        best_split["dataset_right"] = ds2
                        best_split["info_gain"] = curr_info_gain
                        max_info_gain = curr_info_gain
        return best_split
    
    def split(self, dataset, feature_index, threshold):
        ds1, ds2 = [], []
        for row in dataset:
            if row[feature_index] <= threshold:
                ds1.append(row)
            else:
                ds2.append(row)
        return np.array(ds1), np.array(ds2)
    
    def information_gain(self, parent, l_child, r_child, mode="entropy"):
        weight_l = len(l_child) / len(parent)
        weight_r = len(r_child) / len(parent)
        if mode=="gini":
            gain = self.gini_index(parent) - (weight_l*self.gini_index(l_child) + weight_r*self.gini_index(r_child))
        else:
            gain = self.entropy(parent) - (weight_l*self.entropy(l_child) + weight_r*self.entropy(r_child))
        return gain
    
    
    def gini_index(self, y):
        class_labels = np.unique(y)
        gini = 0
        for cls in class_labels:
            p_cls = len(y[y == cls]) / len(y)
            gini += p_cls*p_cls
        return 1 - gini
        
    def calculate_leaf_value(self, Y):
        Y = list(Y)
        return max(Y, key=Y.count)
    
    def fit(self, X, Y):
        dataset = np.concatenate((X, Y), axis=1)
        self.root = self.build_tree(dataset)
    
    def predict(self, X):
        return [self.make_prediction(x, self.root) for x in X]
    
    def make_prediction(self, x, tree):
        if tree.value!=None: 
            return tree.value
        feature_val = x[tree.feature_index]
        if feature_val<=tree.threshold:
            return self.make_prediction(x, tree.left)
        else:
            return self.make_prediction(x, tree.right)

## Fit the model

In [6]:
print(X_reduced.shape)
y_train=y_train.reshape(-1,1)
print(y_train.shape)
classifier = DecisionTreeClassifier(min_samples_split=3, max_depth=2)
classifier.fit(X_reduced[:5000],y_train[:5000])

(18623, 10)
(18623, 1)


## Test the model

In [7]:

Y_pred = classifier.predict(X_reduced1)

right = 0
for i in range(len(y_test)):
    if (y_test[i] == Y_pred[i]):
        right = right + 1
accuracy_score = right/len(Y_pred)
print(accuracy_score)


0.6619002224340642


In [8]:
def calculate_class_wise_accuracy(predictions, labels):
    unique_classes = np.unique(labels)
    class_wise_accuracy = {}
    for class_val in unique_classes:
        class_indices = np.where(labels == class_val)[0]
        class_accuracy = np.sum(predictions[class_indices] == labels[class_indices]) / len(class_indices)
        class_wise_accuracy[class_val] = class_accuracy
    return class_wise_accuracy


class_wise_accuracy = calculate_class_wise_accuracy(np.array(Y_pred), np.array(y_test))

for class_val, accuracy in class_wise_accuracy.items():
    print(f"Accuracy for class {class_val}: {accuracy}")

Accuracy for class 0: 0.32857142857142857
Accuracy for class 1: 0.9145374449339208
Accuracy for class 2: 0.7005813953488372


In [9]:
print(X_reduced.shape)
print(y_train.shape)

(18623, 10)
(18623, 1)


In [11]:
from collections import Counter

def perform_bagging(X_train, y_train, num_samples=5):
    datasets = []
    for _ in range(num_samples):
        indices = np.random.choice(len(X_train), 100, replace=True)
        X_sample, y_sample = X_train[indices], y_train[indices]
        datasets.append((X_sample, y_sample))
    return datasets

def train_decision_trees(datasets):
    trees = []
    for X_sample, y_sample in datasets:
        tree = DecisionTreeClassifier(min_samples_split=3, max_depth=2)
        tree.fit(X_sample, y_sample)
        trees.append(tree)
    return trees

def majority_voting(trees, X_test):
    predictions = []
    for tree in trees:
        prediction = tree.predict(X_test)
        predictions.append(prediction)
    majority_votes = np.array(predictions)
    majority_votes = np.swapaxes(majority_votes, 0, 1)
    final_predictions = []
    for votes in majority_votes:
        vote_count = Counter(votes)
        final_predictions.append(vote_count.most_common(1)[0][0])
    return final_predictions

datasets = perform_bagging(X_reduced[:1000], y_train[:1000])
trees = train_decision_trees(datasets)
predictions = majority_voting(trees, X_reduced1)
total_accuracy = np.mean(predictions == y_test)

class_accuracy = {}
for class_label in np.unique(y_test):
    correct = np.sum((predictions == y_test) & (y_test == class_label))
    total = np.sum(y_test == class_label)
    class_accuracy[class_label] = correct / total

print(f"Total Accuracy: {total_accuracy}")

print(f"Class-wise Accuracy: {class_accuracy}")

Total Accuracy: 0.6431522084524944
Class-wise Accuracy: {0: 0.6775510204081633, 1: 0.9647577092511013, 2: 0.2567829457364341}
